# QC-Py-Cloud-02 : Classification de Texte et Sentiment NLP

[< Retour au sommaire](./QC-Py-00-Sommaire.ipynb) | [Suivant : Risk Parity >](./QC-Py-Cloud-03-Risk-Parity.ipynb)

**Niveau** : Intermediaire | **Duree estimee** : 30 min | **Kernel** : Python 3

## Objectifs

- Comprendre le pipeline NLP : tokenisation, vocabulaire, bag-of-words
- Implementer un classifieur Naive Bayes pour le sentiment financier
- Combiner signaux textuels et indicateurs techniques (modele hybride)
- Deployer l'algorithme sur QuantConnect Cloud via les outils MCP

---

## Partie 1 : Traitement du Langage Naturel pour la Finance

Le traitement du langage naturel (NLP) applique aux marches financiers permet d'extraire
du signal a partir de textes : articles de presse, rapports d'analystes, fils d'actualite.

Le pipeline classique comporte quatre etapes :

1. **Collecte** des donnees textuelles (news, filings, social media)
2. **Pretraitement** : tokenisation, nettoyage, construction du vocabulaire
3. **Vectorisation** : representation numerique (bag-of-words, TF-IDF)
4. **Classification** : prediction du sentiment (positif, negatif, neutre)

### Approche Bag-of-Words

La representation bag-of-words ignore l'ordre des mots et se concentre sur leur frequence
d'apparition. Pour le domaine financier, un vocabulaire restreint (500-2000 mots) capture
deja l'essentiel du signal sentimental.

In [1]:
# Demonstration : construction d'un vocabulaire financier
from collections import Counter

sample_headlines = [
    "AAPL surges on heavy volume after earnings beat",
    "MSFT declines on weak guidance outlook",
    "GOOGL trades mixed amid market uncertainty",
    "AMZN rallies strongly on Prime Day results",
    "NVDA plummets on high volume after downgrade",
    "AAPL gains ground following product launch",
    "MSFT profit exceeds expectations significantly",
    "GOOGL drops sharply on antitrust concerns",
]

all_words = []
for h in sample_headlines:
    words = h.lower().split()
    all_words.extend(words)

word_counts = Counter(all_words)
top_20 = word_counts.most_common(20)

print("Top 20 mots du vocabulaire financier :")
for word, count in top_20:
    print(f"  {word:20s} {count}")

print(f"\nVocabulaire total : {len(word_counts)} mots uniques")


Top 20 mots du vocabulaire financier :
  on                   5
  aapl                 2
  volume               2
  after                2
  msft                 2
  googl                2
  surges               1
  heavy                1
  earnings             1
  beat                 1
  declines             1
  weak                 1
  guidance             1
  outlook              1
  trades               1
  mixed                1
  amid                 1
  market               1
  uncertainty          1
  amzn                 1

Vocabulaire total : 42 mots uniques


Le vocabulaire issu des titres d'actualite financiere contient a la fois des mots
neutres et des mots porteurs de sentiment (surges, plummets, beat, downgrade).
Le classifieur Naive Bayes apprend a associer ces mots aux mouvements de prix.

### Modele Naive Bayes Multinomial

Le classifieur MultinomialNB est adapte aux donnees textuelles representees en
bag-of-words. Il estime P(classe|mot) via le theoreme de Bayes.

In [2]:
# Simulation : scoring Naive Bayes sur des titres
import numpy as np

positive_words = {"surges", "rallies", "gains", "beat", "profit", "growth",
                  "strongly", "exceeds", "significantly", "strong"}
negative_words = {"declines", "plummets", "drops", "weak", "downgrade",
                  "concerns", "sharply", "uncertainty", "antitrust"}

def sentiment_score(headline):
    words = set(headline.lower().split())
    pos = len(words & positive_words)
    neg = len(words & negative_words)
    total = pos + neg
    if total == 0:
        return 0.0
    return (pos - neg) / total

for h in sample_headlines:
    score = sentiment_score(h)
    label = "BULL" if score > 0 else "BEAR" if score < 0 else "NEUTRAL"
    print(f"  [{label:6s}] {score:+.2f}  {h}")


  [BULL  ] +1.00  AAPL surges on heavy volume after earnings beat
  [BEAR  ] -1.00  MSFT declines on weak guidance outlook
  [BEAR  ] -1.00  GOOGL trades mixed amid market uncertainty
  [BULL  ] +1.00  AMZN rallies strongly on Prime Day results
  [BEAR  ] -1.00  NVDA plummets on high volume after downgrade
  [BULL  ] +1.00  AAPL gains ground following product launch
  [BULL  ] +1.00  MSFT profit exceeds expectations significantly
  [BEAR  ] -1.00  GOOGL drops sharply on antitrust concerns


---

## Partie 2 : Algorithme QuantConnect - Classification de Texte

L'algorithme combine :
- **Signal textuel** : vocabulaire bag-of-words + Naive Bayes Multinomial
- **Signal technique** : RSI, ratio EMA, momentum
- **Score hybride** : 50% sentiment + 50% technique

L'univers comprend 5 actions tech (AAPL, MSFT, GOOGL, AMZN, NVDA) avec
un rebalancement hebdomadaire.

In [3]:
# Code source de l'algorithme - pret pour deploiement QC Cloud
qc_code = '''# region imports
from AlgorithmImports import *
# endregion

class MLTextClassificationAlgorithm(QCAlgorithm):
    """
    NLP Text Classification Strategy.

    Strategy:
    - Use simulated news sentiment for trading decisions
    - Train text classifier (Naive Bayes) on sentiment labels
    - Features: Bag-of-words, TF-IDF from news headlines
    - Target: Next-day price direction (up/down)
    - Combine sentiment with technical indicators for hybrid model
    """

    def Initialize(self):
        self.SetStartDate(2020, 1, 1)
        self.SetCash(100000)

        # Assets
        self.tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
        self.symbols = {}
        for ticker in self.tickers:
            self.symbols[ticker] = self.AddEquity(ticker, Resolution.DAILY).Symbol

        # Text classification parameters
        self.lookback = 60
        self.rebalance_freq = 5
        self.vocabulary_size = 1000
        self.sentiment_threshold = 0.6

        # Rebalance schedule
        self.Schedule.On(self.DateRules.Every(DayOfWeek.Monday),
                         self.TimeRules.AfterMarketOpen("SPY", 30),
                         self.Rebalance)

        # Train model bi-weekly
        self.Schedule.On(self.DateRules.Every(DayOfWeek.Monday),
                         self.TimeRules.AfterMarketOpen("SPY", 30),
                         self.TrainModel)

        self.model = None
        self.vectorizer = None
        self.scaler = None

    def SimulateNewsHeadlines(self, ticker, prices, volumes):
        """Simulate news headlines based on price/volume action.

        In production, this would use real news APIs.
        """
        headlines = []

        # Calculate recent changes
        returns = prices.pct_change()
        vol_changes = volumes.pct_change()

        for i in range(len(prices)):
            if i == 0:
                headlines.append("Market opens steady")
                continue

            ret = returns.iloc[i]
            vol_change = vol_changes.iloc[i] if i < len(vol_changes) else 0

            # Generate headline based on market action
            if ret > 0.03:
                if vol_change > 0.2:
                    headlines.append(f"{ticker} surges on heavy volume")
                else:
                    headlines.append(f"{ticker} rallies strongly")
            elif ret > 0.01:
                headlines.append(f"{ticker} gains ground")
            elif ret > -0.01:
                headlines.append(f"{ticker} trades mixed")
            elif ret > -0.03:
                headlines.append(f"{ticker} declines")
            else:
                if vol_change > 0.2:
                    headlines.append(f"{ticker} plummets on high volume")
                else:
                    headlines.append(f"{ticker} drops sharply")

        return headlines

    def CreateVocabulary(self, headlines):
        """Create vocabulary from headlines."""
        # Simple word tokenization
        all_words = []
        for headline in headlines:
            words = headline.lower().split()
            all_words.extend(words)

        # Count word frequencies
        from collections import Counter
        word_counts = Counter(all_words)

        # Get top words
        vocabulary = [word for word, _ in word_counts.most_common(self.vocabulary_size)]
        return {word: idx for idx, word in enumerate(vocabulary)}

    def HeadlineToFeatures(self, headline, vocabulary):
        """Convert headline to feature vector (bag-of-words)."""
        words = headline.lower().split()
        features = np.zeros(self.vocabulary_size)

        for word in words:
            if word in vocabulary:
                features[vocabulary[word]] = 1

        return features

    def TrainModel(self):
        """Train text classification model."""
        self.Debug("Training text classification model...")

        all_features = []
        all_targets = []

        for ticker in self.tickers:
            history = self.History(self.symbols[ticker], self.lookback, Resolution.Daily)

            if history.empty or len(history) < self.lookback:
                continue

            closes = history['close']
            volumes = history['volume']

            # Simulate headlines
            headlines = self.SimulateNewsHeadlines(ticker, closes, volumes)

            # Create vocabulary
            self.vocabulary = self.CreateVocabulary(headlines)

            # Create features and targets
            for i in range(1, len(headlines) - 1):
                features = self.HeadlineToFeatures(headlines[i], self.vocabulary)

                # Target: next day direction
                next_return = (closes.iloc[i + 1] - closes.iloc[i]) / closes.iloc[i]
                target = 1 if next_return > 0 else 0

                all_features.append(features)
                all_targets.append(target)

        if len(all_features) < 20:
            return

        # Train Naive Bayes classifier
        from sklearn.naive_bayes import MultinomialNB
        from sklearn.preprocessing import StandardScaler

        X = np.array(all_features)
        y = np.array(all_targets)

        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X)

        self.model = MultinomialNB()
        self.model.fit(X_scaled + 1, y)  # NB requires non-negative

        self.Debug("Text classification model trained.")

    def GetSentiment(self, ticker, history):
        """Get sentiment score from recent headlines."""
        if self.model is None or self.vocabulary is None:
            return 0.5

        closes = history['close']
        volumes = history['volume']

        # Simulate recent headlines
        headlines = self.SimulateNewsHeadlines(ticker, closes, volumes)

        # Get sentiment from last few headlines
        recent_headlines = headlines[-5:]
        sentiments = []

        for headline in recent_headlines:
            features = self.HeadlineToFeatures(headline, self.vocabulary)
            features_scaled = self.scaler.transform([features + 1])[0]
            prob = self.model.predict_proba([features_scaled])[0][1]
            sentiments.append(prob)

        return np.mean(sentiments)

    def GetTechnicalFeatures(self, history):
        """Get technical features for hybrid model."""
        closes = history['close']

        # RSI
        delta = closes.diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))

        # EMA ratio
        ema20 = closes.ewm(span=20).mean()
        ema50 = closes.ewm(span=50).mean()
        ema_ratio = ema20.iloc[-1] / ema50.iloc[-1]

        # Momentum
        momentum = (closes.iloc[-1] / closes.iloc[-5] - 1) if len(closes) >= 5 else 0

        return {
            'rsi': rsi.iloc[-1],
            'ema_ratio': ema_ratio,
            'momentum': momentum
        }

    def Rebalance(self):
        """Rebalance based on sentiment + technical analysis."""
        if self.model is None:
            return

        predictions = {}

        for ticker in self.tickers:
            try:
                history = self.History(self.symbols[ticker], 60, Resolution.Daily)

                if history.empty:
                    continue

                # Get sentiment
                sentiment = self.GetSentiment(ticker, history)

                # Get technical features
                tech = self.GetTechnicalFeatures(history)

                # Hybrid score: sentiment + technical
                score = (
                    sentiment * 0.5 +
                    (1 if tech['rsi'] < 30 else 0 if tech['rsi'] > 70 else 0.5) * 0.2 +
                    (1 if tech['ema_ratio'] > 1 else 0) * 0.15 +
                    (1 if tech['momentum'] > 0 else 0) * 0.15
                )

                predictions[ticker] = score

            except:
                continue

        if not predictions:
            return

        # Sort by score
        sorted_preds = sorted(predictions.items(), key=lambda x: x[1], reverse=True)

        # Liquidate
        self.Liquidate()

        # Enter long positions
        count = 0
        for ticker, score in sorted_preds:
            if score > self.sentiment_threshold and count < 3:
                self.SetHoldings(self.symbols[ticker], 0.3)
                count += 1

        self.Debug(f"Positions: {count}, Best score: {sorted_preds[0][1]:.2f}" if sorted_preds else "No positions")

'''

print(f"Algorithme charge : {len(qc_code)} caracteres")
print(f"Lignes de code : {len(qc_code.split(chr(10)))}")
print("Classe : MLTextClassificationAlgorithm")
print("Univers : AAPL, MSFT, GOOGL, AMZN, NVDA")
print("Modele : Naive Bayes Multinomial + indicateurs techniques")
print("Rebalancement : hebdomadaire (lundi)")


Algorithme charge : 8555 caracteres
Lignes de code : 263
Classe : MLTextClassificationAlgorithm
Univers : AAPL, MSFT, GOOGL, AMZN, NVDA
Modele : Naive Bayes Multinomial + indicateurs techniques
Rebalancement : hebdomadaire (lundi)


### Architecture de l'algorithme

| Composant | Methode | Description |
|-----------|---------|-------------|
| Vocabulaire | `CreateVocabulary()` | Top 1000 mots par frequence |
| Vectorisation | `HeadlineToFeatures()` | Bag-of-words binaire |
| Entrainement | `TrainModel()` | MultinomialNB bi-hebdomadaire |
| Sentiment | `GetSentiment()` | Moyenne des probabilites recentes |
| Technique | `GetTechnicalFeatures()` | RSI, EMA ratio, momentum |
| Score hybride | `Rebalance()` | 50% sentiment + 50% technique |

---

## Partie 3 : Deploiement via MCP QuantConnect

Le workflow MCP pour deployer cet algorithme sur QC Cloud suit les etapes
standard : creation du projet, ecriture du code, compilation, backtest.

In [4]:
# Workflow MCP QuantConnect
print("Workflow de deploiement MCP :")
print()
print("1. create_project(name='ML-TextClassification')")
print("   -> Cree un nouveau projet dans l'organisation QC")
print()
print("2. update_file_contents(projectId, name='main.py', content=qc_code)")
print("   -> Charge le code source dans le projet")
print()
print("3. create_compile(projectId)")
print("   -> Compile l'algorithme sur l'infrastructure QC Cloud")
print()
print("4. create_backtest(projectId, compileId, name='ML-TextClass-v1')")
print("   -> Lance le backtest (periode 2020-2026)")
print()
print("5. read_backtest(projectId, backtestId)")
print("   -> Recupere les metriques : Sharpe, CAGR, MaxDD, alpha")
print()
print("Note : Deployez l'algorithme via MCP pour obtenir les resultats reels.")


Workflow de deploiement MCP :

1. create_project(name='ML-TextClassification')
   -> Cree un nouveau projet dans l'organisation QC

2. update_file_contents(projectId, name='main.py', content=qc_code)
   -> Charge le code source dans le projet

3. create_compile(projectId)
   -> Compile l'algorithme sur l'infrastructure QC Cloud

4. create_backtest(projectId, compileId, name='ML-TextClass-v1')
   -> Lance le backtest (periode 2020-2026)

5. read_backtest(projectId, backtestId)
   -> Recupere les metriques : Sharpe, CAGR, MaxDD, alpha

Note : Deployez l'algorithme via MCP pour obtenir les resultats reels.


---

## Partie 4 : Resultats attendus et interpretation

### Metriques cles a observer

| Metrique | Description | Benchmark |
|----------|-------------|-----------|
| Sharpe Ratio | Rendement ajuste au risque | > 0.5 |
| CAGR | Rendement annuel compose | > 8% |
| Max Drawdown | Perte maximale depuis un pic | < 30% |
| Win Rate | % de trades positifs | > 50% |
| Alpha | Rendement excessif vs benchmark | > 0% |

### Points d'attention

- Le vocabulaire simule peut ne pas capturer toute la richesse des vraies news
- Le modele hybride (sentiment + technique) est plus robuste que le sentiment seul
- Le classifieur Naive Bayes est un bon point de depart mais peut etre ameliore

In [5]:
# Placeholder pour les resultats du backtest QC Cloud
print("Resultats du backtest (a deployer via MCP) :")
print()
print("  Algorithme    : MLTextClassificationAlgorithm")
print("  Periode       : 2020-01-01 -> 2026-01-01")
print("  Capital initial : 100 000 USD")
print("  Univers        : AAPL, MSFT, GOOGL, AMZN, NVDA")
print()
print("  Metriques a recuperer :")
print("    - Sharpe Ratio")
print("    - CAGR")
print("    - Max Drawdown")
print("    - Total Trades")
print("    - Win Rate")
print()
print("Note : Deployez via MCP pour obtenir les resultats reels.")


Resultats du backtest (a deployer via MCP) :

  Algorithme    : MLTextClassificationAlgorithm
  Periode       : 2020-01-01 -> 2026-01-01
  Capital initial : 100 000 USD
  Univers        : AAPL, MSFT, GOOGL, AMZN, NVDA

  Metriques a recuperer :
    - Sharpe Ratio
    - CAGR
    - Max Drawdown
    - Total Trades
    - Win Rate

Note : Deployez via MCP pour obtenir les resultats reels.


---

## Conclusion

Ce notebook a demontre le pipeline complet de classification de texte applique
au trading quantitatif :

1. **Pretraitement NLP** : tokenisation, vocabulaire, bag-of-words
2. **Modele Naive Bayes** : classification probabiliste du sentiment
3. **Approche hybride** : combinaison sentiment + indicateurs techniques
4. **Deploiement Cloud** : algorithme pret pour QC Cloud via MCP

| Concept | Outil QC | Utilisation |
|---------|----------|-------------|
| Vocabulaire | `History()` | Construction du lexique |
| Bag-of-Words | `numpy` | Vectorisation des titres |
| Naive Bayes | `sklearn` | Classification sentiment |
| RSI / EMA | Indicators API | Signal technique |
| Rebalancement | `Schedule.On()` | Execution hebdomadaire |

[< Retour au sommaire](./QC-Py-00-Sommaire.ipynb) | [Suivant : Risk Parity >](./QC-Py-Cloud-03-Risk-Parity.ipynb)